# Verificación de `nflreadpy` como fuente de datos — WR semanal

## Qué es esto y por qué existe

Este proyecto predice, para cada receptor (WR) de la NFL, cuánto va a producir la próxima
semana (recepciones, yardas, touchdowns). Para eso necesitamos datos: quién es cada jugador,
qué partidos hay, qué hizo cada quien cada semana, quién es titular, y el detalle jugada por
jugada.

Todos esos datos vienen de **`nflverse`**, un proyecto de código abierto mantenido por la
comunidad (no una empresa con contrato ni soporte). Se puede leer desde Python con dos
librerías distintas: `nfl_data_py` (la que usaba el pipeline heredado, hoy descontinuada) y
`nflreadpy` (su sucesora activa, la que vamos a usar de aquí en adelante).

**Antes de construir el pipeline nuevo sobre `nflreadpy`, hay que comprobar que sea confiable**
— no basta con que "funcione", hay que revisar si los datos que entrega están completos,
correctos, y disponibles a tiempo. Eso es lo que hace este notebook, sobre las 5 fuentes
concretas que el pipeline va a usar:

| Fuente (función de `nflreadpy`) | Qué responde |
|---|---|
| `load_players()` | ¿Quién es cada jugador? (identidad) |
| `load_schedules()` | ¿Qué partidos hay, cuándo, con qué contexto (Vegas, clima)? |
| `load_player_stats()` | ¿Qué produjo cada jugador, cada semana? (el target a predecir) |
| `load_pbp()` | ¿Qué pasó en cada jugada del partido? (para calcular `target_share`) |
| `load_depth_charts()` | ¿Quién es titular esta semana? |

Si cualquiera de estas 5 piezas tiene datos incompletos, duplicados, o con retraso, el modelo
entero hereda ese problema sin que nadie lo note — por eso se revisa cada una por separado, con
evidencia, antes de construir nada encima.

## Cómo usar este notebook

Se ejecuta dentro del contenedor Docker del proyecto (`docker compose up python`, servicio
`python`), que ya tiene `nflreadpy` instalado. La variable `SEASONS` (más abajo, antes de la
Parte 2) es el único parámetro que normalmente cambiarías — hoy se usan 2024 y 2025 (dos
temporadas completas y ya cerradas), pero se puede correr sobre cualquier otro rango de años sin
tocar el resto del código.

Los resultados de fondo ya están incorporados a la documentación permanente del proyecto:
[`docs/data/DATASHEET.md`](../../docs/data/DATASHEET.md) (sección "Auditoría de `nflreadpy` por
fuente") y [`docs/HALLAZGOS.md`](../../docs/HALLAZGOS.md). Este notebook es la evidencia
ejecutable detrás de esos documentos — si algo cambia (nueva versión de `nflreadpy`, más años),
se vuelve a correr y se compara.

## Parte 1 — ¿La temporada en curso llega a tiempo?

Cada semana, alguien del lado de `nflverse` tiene que armar y publicar la tabla resumen de
estadísticas de esa semana. Ese proceso puede atrasarse unos días respecto al partido real. Si
nuestro pipeline depende de esa tabla y se atrasa, no podríamos generar la predicción de la
semana en curso a tiempo — tendríamos que esperar.

Ya sabíamos (documentado antes) que la librería vieja, `nfl_data_py`, tiene justo ese problema:
su función `import_weekly_data()` regresa error para la temporada que está en curso ahora mismo.
La pregunta que responde esta sección: **¿la función equivalente de `nflreadpy`,
`load_player_stats(summary_level="week")`, tiene el mismo problema, o no?**

Se prueba llamando a ambas el mismo día, para la misma temporada, y comparando qué regresa cada
una.

In [1]:
import datetime
import nfl_data_py as nfldp
import nflreadpy as nfl

print("Hoy:", datetime.date.today())

print()
print("=== nfl_data_py.import_weekly_data([2026]) — la libreria vieja ===")
try:
    wk_old = nfldp.import_weekly_data([2026])
    print("filas:", len(wk_old), "| semanas:", sorted(wk_old["week"].unique().tolist()))
except Exception as e:
    print("ERROR:", type(e).__name__, "-", str(e)[:200])

print()
print("=== nflreadpy.load_player_stats([2026], summary_level='week') — la libreria nueva ===")
ws_2026 = nfl.load_player_stats([2026], summary_level="week")
print("filas:", ws_2026.height, "| semanas:", sorted(ws_2026["week"].unique().to_list()))

Hoy: 2026-09-29

=== nfl_data_py.import_weekly_data([2026]) — la libreria vieja ===


ERROR: HTTPError - HTTP Error 404: Not Found

=== nflreadpy.load_player_stats([2026], summary_level='week') — la libreria nueva ===
filas: 3339 | semanas: [1, 2, 3]


**Resultado:** `import_weekly_data` falla (error 404) — la tabla resumen de la temporada en
curso todavía no existe del lado viejo. `load_player_stats`, en cambio, sí trae datos, y llegan
hasta la última semana que ya se jugó completa. Es decir: **la librería nueva no hereda el
retraso de la vieja**, aunque ambas piden "lo mismo" conceptualmente.

Esto simplifica el diseño del pipeline: ya no hace falta el plan B de calcular todo a mano desde
el detalle de jugadas (`load_pbp`) solo para evitar el retraso — se puede usar
`load_player_stats` directamente, tanto para historia como para la semana en curso.

## Parte 2 — Calidad de cada fuente, con volumen real (2 temporadas completas)

La Parte 1 solo probó 3 semanas de datos. Eso alcanza para responder "¿llega a tiempo?", pero no
alcanza para saber si los datos están *bien* — huecos raros, duplicados, identidades mezcladas,
etc. solo se notan con volumen real. Por eso aquí se descargan **dos temporadas completas y ya
cerradas** (2024 y 2025 — cerradas para no mezclar el análisis con el tema del retraso de la
Parte 1) y se revisa cada fuente con el mismo criterio de 6 dimensiones de calidad que ya se usó
antes en este proyecto para auditar el archivo de ADP (exactitud, completitud, consistencia,
validez, vigencia, unicidad — ver el Datasheet).

In [2]:
import polars as pl

SEASONS = [2024, 2025]  # cambiar aqui para correr esto sobre otro rango de anios

### Fuente 1 — `load_players()`: identidad de jugador

Esta es la tabla maestra de todos los jugadores que ha habido en la NFL — cada uno con un
identificador único, `gsis_id`. La necesitamos para saber, sin ambigüedad, "quién es quién" al
cruzar las demás fuentes. Ya vimos antes, en el archivo de ADP del repo heredado, que un mismo
`gsis_id` podía aparecer repetido en 2 o 3 jugadores distintos — eso hacía que sus estadísticas
se mezclaran. Aquí se revisa si `load_players()` tiene el mismo problema.

In [3]:
players = nfl.load_players()
print("filas (todos los jugadores historicos):", players.height, "| columnas:", players.width)

gsis = players.filter(players["gsis_id"].is_not_null())
dup = gsis["gsis_id"].value_counts().filter(pl.col("count") > 1)
print("gsis_id no nulos:", gsis.height, "/", players.height)
print("gsis_id que se repiten en mas de una fila (0 es lo esperado):", dup.height)

cols_clave = ["gsis_id", "display_name", "position", "birth_date", "draft_year", "draft_round"]
for c in cols_clave:
    pct = round(players[c].null_count() / players.height * 100, 2)
    print(f"  % de filas sin dato en '{c}': {pct}%")

filas (todos los jugadores historicos): 24834 | columnas: 39
gsis_id no nulos: 24834 / 24834
gsis_id que se repiten en mas de una fila (0 es lo esperado): 0
  % de filas sin dato en 'gsis_id': 0.0%
  % de filas sin dato en 'display_name': 0.0%
  % de filas sin dato en 'position': 0.0%
  % de filas sin dato en 'birth_date': 0.13%
  % de filas sin dato en 'draft_year': 49.7%
  % de filas sin dato en 'draft_round': 49.7%


**Lectura:** cero colisiones de `gsis_id` en 24,834 jugadores — a diferencia del archivo de ADP,
aquí cada identificador corresponde a un único jugador. Esto confirma una decisión ya tomada:
usar `load_players()` para resolver identidad, no el archivo de ADP.

Sobre los nulos: `draft_year`/`draft_round` faltan en ~50% de los jugadores. Antes de alarmarse
— **¿tiene sentido ese hueco?** Sí: un jugador que entró a la liga sin ser drafteado
simplemente no tiene año/ronda de draft, no es que falte el dato, es que ese dato no aplica para
esa persona. Es un "nulo estructural", no un error — se documenta así y no requiere ninguna
acción de limpieza.

### Fuente 2 — `load_schedules()`: el calendario de partidos

Trae, para cada partido, cuándo se jugó, quién contra quién, y contexto del ambiente del juego:
la línea de Vegas (`spread_line`/`total_line`, cuánto esperaba el mercado de apuestas que
anotara cada equipo) y el clima (`temp`/`wind`). Este contexto es lo que en otra parte de este
proyecto se probó como posible feature para el modelo (con resultado: efecto casi nulo para WR,
ya documentado).

In [4]:
sched = nfl.load_schedules(SEASONS)
print("filas:", sched.height, "| si solo fuera temporada regular esperariamos:", 272 * len(SEASONS))
print("semanas presentes:", sorted(sched["week"].unique().to_list()))
print("tipos de partido presentes (game_type):", sorted(sched["game_type"].unique().to_list()))

for c in ["spread_line", "total_line", "temp", "wind"]:
    pct = round(sched[c].null_count() / sched.height * 100, 2)
    print(f"  % de partidos sin dato en '{c}': {pct}%")

filas: 570 | si solo fuera temporada regular esperariamos: 544
semanas presentes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
tipos de partido presentes (game_type): ['CON', 'DIV', 'REG', 'SB', 'WC']
  % de partidos sin dato en 'spread_line': 0.0%
  % de partidos sin dato en 'total_line': 0.0%
  % de partidos sin dato en 'temp': 34.74%
  % de partidos sin dato en 'wind': 34.74%


**Lectura:** salen más partidos de los que esperábamos (272 por temporada regular) porque
`load_schedules()` trae *también* los playoffs por default (semanas 19 a 22: Wild Card,
Divisional, Conferencia, Super Bowl). Eso no es un error de la fuente — es que hay que pedirle
explícitamente "solo temporada regular" (`game_type == "REG"`) al construir el pipeline, o de lo
contrario un análisis de "promedio por semana" mezclaría partidos de playoffs sin querer.

`spread_line`/`total_line` (Vegas) están completos, sin huecos. `temp`/`wind` faltan en ~35% de
los partidos — de nuevo, ¿tiene sentido? Sí: los partidos bajo techo (domo) no tienen clima real
que reportar. No es un hueco de datos, es un "no aplica".

### Fuente 3 — `load_player_stats(summary_level="week")`: lo que produjo cada jugador, cada semana

Esta es la fuente más importante del pipeline: de aquí sale literalmente el número que se quiere
predecir (recepciones, yardas, touchdowns de cada WR, cada semana). Si esta tabla tiene
duplicados o huecos raros, el modelo aprendería de datos incorrectos sin que se note.

In [5]:
ws = nfl.load_player_stats(SEASONS, summary_level="week")
wr = ws.filter(ws["position"] == "WR")
print("filas totales (todas las posiciones):", ws.height, "| filas solo WR:", wr.height)

# Cada jugador deberia aparecer una sola vez por temporada y semana.
# Si aparece 2+ veces, algo esta mal (o se sumarian mal sus stats, o se duplicarian filas).
dup = ws.group_by(["player_id", "season", "week"]).len().filter(pl.col("len") > 1)
print("combinaciones (jugador, temporada, semana) que aparecen mas de una vez (0 es lo esperado):", dup.height)

cols_wr = ["receptions", "receiving_yards", "receiving_tds", "targets",
           "target_share", "racr", "air_yards_share", "wopr", "receiving_epa"]
for c in cols_wr:
    pct = round(wr[c].null_count() / wr.height * 100, 2)
    print(f"  % de filas WR sin dato en '{c}': {pct}%")

filas totales (todas las posiciones): 38405 | filas solo WR: 5195
combinaciones (jugador, temporada, semana) que aparecen mas de una vez (0 es lo esperado): 0
  % de filas WR sin dato en 'receptions': 0.0%
  % de filas WR sin dato en 'receiving_yards': 0.0%
  % de filas WR sin dato en 'receiving_tds': 0.0%
  % de filas WR sin dato en 'targets': 0.0%
  % de filas WR sin dato en 'target_share': 0.0%
  % de filas WR sin dato en 'racr': 15.42%
  % de filas WR sin dato en 'air_yards_share': 0.0%
  % de filas WR sin dato en 'wopr': 0.0%
  % de filas WR sin dato en 'receiving_epa': 15.23%


**Lectura:** cero filas duplicadas — cada WR aparece exactamente una vez por semana, que es
justo lo que se necesita para entrenar un modelo sin fugas. Las columnas que más importan
(recepciones, yardas, `target_share`) están completas al 100%.

`racr` y `receiving_epa` sí tienen huecos (~15%). Antes de decidir qué hacer, hay que entender
por qué — se investiga en la siguiente celda.

In [6]:
null_racr = wr.filter(pl.col("racr").is_null())
print("filas WR con 'racr' vacio:", null_racr.height)
print("  de esas, jugadores que esa semana tuvieron 0 targets (targets == 0):",
      null_racr.filter(pl.col("targets") == 0).height)
print("  de esas, con targets > 0 (esto SI seria raro de explicar):",
      null_racr.filter(pl.col("targets") > 0).height)

filas WR con 'racr' vacio: 801
  de esas, jugadores que esa semana tuvieron 0 targets (targets == 0): 796
  de esas, con targets > 0 (esto SI seria raro de explicar): 5


**Conclusión:** 796 de 801 filas (99.4%) con `racr` vacío son de jugadores que esa semana no
tuvieron ni un solo target — tiene sentido matemático que ese cálculo salga vacío si no hubo
ninguna jugada de la que partir (es "no aplica", igual que el clima bajo techo). Solo quedan 5
filas (0.1% del total de filas WR) sin una explicación clara — un residuo tan pequeño que se
documenta y se revisa más adelante si esas columnas llegan a usarse como feature, pero no
bloquea nada hoy.

**Otro hallazgo, mismo patrón que el calendario:** `load_player_stats()` también mezcla
playoffs por default. Se revisa antes de dar por buena la fuente.

In [7]:
print("season_type presentes en las stats semanales:", ws["season_type"].unique().to_list())
print("semanas presentes (deberian llegar hasta 22 si incluye playoffs):",
      sorted(ws["week"].unique().to_list()))

season_type presentes en las stats semanales: ['REG', 'POST']
semanas presentes (deberian llegar hasta 22 si incluye playoffs): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]


### Fuente 4 — `load_pbp()`: el detalle de cada jugada

Esta fuente trae una fila por cada jugada de cada partido (no por jugador-semana) — se usa para
calcular `target_share` (qué porcentaje de los pases de su equipo recibió cada jugador) desde
cero, algo que `load_player_stats` no da directamente por jugada.

In [8]:
pbp = nfl.load_pbp(SEASONS)
print("filas (una por jugada):", pbp.height)
for c in ["receiver_id", "receiver_player_name", "pass_attempt", "air_yards"]:
    pct = round(pbp[c].null_count() / pbp.height * 100, 2)
    print(f"  % de jugadas sin dato en '{c}': {pct}%")
print("season_type presentes:", pbp["season_type"].unique().to_list())

filas (una por jugada): 98263
  % de jugadas sin dato en 'receiver_id': 61.88%
  % de jugadas sin dato en 'receiver_player_name': 63.94%
  % de jugadas sin dato en 'pass_attempt': 3.08%
  % de jugadas sin dato en 'air_yards': 62.4%
season_type presentes: ['POST', 'REG']


**Lectura:** ~62% de las jugadas no tienen receptor — tiene sentido, porque un acarreo, una
patada o un despeje no involucran a ningún receptor. Es el mismo tipo de "no aplica"
estructural que ya vimos en las otras fuentes, no un problema.

**Y otra vez el mismo patrón de playoffs:** `load_pbp()` también trae `POST` junto con `REG`.
Se filtra igual que en las otras dos fuentes.

### Fuente 5 — `load_depth_charts()`: quién es titular esta semana

Aquí sí apareció algo que **no** se explica con "no aplica" — el único hallazgo real de calidad
de esta auditoría. Se muestra primero el síntoma, tal como se ve a simple vista, y luego se
investiga hasta encontrar la causa real.

In [9]:
depth = nfl.load_depth_charts(SEASONS)
print("filas:", depth.height)
for c in ["position", "depth_team", "pos_abb"]:
    pct = round(depth[c].null_count() / depth.height * 100, 2)
    print(f"  % de filas sin dato en '{c}': {pct}%")

filas: 591527
  % de filas sin dato en 'position': 93.69%
  % de filas sin dato en 'depth_team': 93.69%
  % de filas sin dato en 'pos_abb': 6.31%


A primera vista, un ~94% de huecos en `position`/`depth_team` se ve como un problema grave de
calidad. Pero antes de concluir eso, hay que preguntarse lo mismo que en las otras fuentes:
**¿ese hueco tiene una explicación estructural?** Para saberlo, se pide cada temporada por
separado (en vez de las dos juntas) y se comparan las columnas que trae cada una:

In [10]:
for yr in [2022, 2023, 2024, 2025, 2026]:
    d = nfl.load_depth_charts([yr])
    print(f"{yr}: filas={d.height} | columnas={d.columns}")

2022: filas=37780 | columnas=['season', 'club_code', 'week', 'game_type', 'depth_team', 'last_name', 'first_name', 'football_name', 'formation', 'gsis_id', 'jersey_number', 'position', 'elias_id', 'depth_position', 'full_name']
2023: filas=37327 | columnas=['season', 'club_code', 'week', 'game_type', 'depth_team', 'last_name', 'first_name', 'football_name', 'formation', 'gsis_id', 'jersey_number', 'position', 'elias_id', 'depth_position', 'full_name']
2024: filas=37312 | columnas=['season', 'club_code', 'week', 'game_type', 'depth_team', 'last_name', 'first_name', 'football_name', 'formation', 'gsis_id', 'jersey_number', 'position', 'elias_id', 'depth_position', 'full_name']


2025: filas=554215 | columnas=['dt', 'team', 'player_name', 'espn_id', 'gsis_id', 'pos_grp_id', 'pos_grp', 'pos_id', 'pos_name', 'pos_abb', 'pos_slot', 'pos_rank']
2026: filas=581172 | columnas=['dt', 'team', 'player_name', 'espn_id', 'gsis_id', 'pos_grp_id', 'pos_grp', 'pos_id', 'pos_name', 'pos_abb', 'pos_slot', 'pos_rank']


**La causa real:** no es un hueco de datos — es un **cambio de esquema** que nadie documentó.
Las temporadas 2022 a 2024 traen columnas `season`/`week`/`game_type`/`position`/`depth_team`
(un formato "histórico", ya reconstruido por semana). **A partir de 2025 — y se confirmó que
también aplica a 2026, la temporada que está en curso ahora mismo — `nflreadpy` cambia a un
formato completamente distinto**: `dt`/`team`/`pos_grp`/`pos_rank`/`pos_abb`, que ni siquiera
tiene columnas de temporada o semana.

Cuando se piden los dos años juntos (`load_depth_charts([2024, 2025])`), la librería junta ambos
formatos en una sola tabla — las filas de 2025 simplemente no tienen las columnas del formato
viejo (por eso salen "vacías"), y viceversa. El dato no falta: vive en una columna con otro
nombre.

**Por qué esto importa para el pipeline:** los notebooks heredados
(`weekly/wr/notebooks/_reference/wrs_rec_tds.ipynb` y sus hermanos) usan `position`/`depth_team`
del formato viejo para identificar a los 3 titulares de cada equipo. Si el pipeline nuevo llama
a `load_depth_charts()` para la temporada en curso (2026) esperando esas mismas columnas, va a
fallar o — peor — va a producir una tabla llena de vacíos sin ningún error visible. El pipeline
nuevo necesita reconocer qué formato le tocó según el año y traducir uno al otro antes de
usarlo. Este hallazgo ya quedó documentado en `docs/HALLAZGOS.md` y en el Datasheet.

## Resumen — qué se puede usar tal cual, y qué necesita trabajo extra

| Fuente | ¿Confiable tal cual? | Qué hacer en el pipeline |
|---|---|---|
| `load_players()` | Sí — identidad limpia, sin colisiones | Usar para identidad, en vez del archivo de ADP |
| `load_schedules()` | Sí, con un ajuste | Filtrar `game_type == "REG"` para no mezclar playoffs |
| `load_player_stats(summary_level="week")` | Sí, con un ajuste | Filtrar `season_type == "REG"` (mismo problema que el calendario); incluye la semana en curso sin retraso |
| `load_pbp()` | Sí, con un ajuste | Filtrar `season_type == "REG"`; los huecos que tiene son esperados |
| `load_depth_charts()` | No tal cual | Construir un adaptador que traduzca el formato viejo (≤2024) y el nuevo (2025+) a las mismas columnas antes de usarlos juntos |

Los 4 filtros/adaptadores de esta tabla ya están implementados en `weekly/wr/src/data.py`
(excepto `load_depth_charts`, todavía pendiente).

Documentación completa, con la evidencia incorporada al proyecto:
[`../../docs/data/DATASHEET.md`](../../docs/data/DATASHEET.md) ·
[`../../docs/HALLAZGOS.md`](../../docs/HALLAZGOS.md)